# GSV-Math Backend — One-Click Demo Server

**How to use:**
1. Make sure Runtime → Change runtime type → **T4 GPU** is selected
2. Fill in your `NGROK_TOKEN` and `HF_TOKEN` in Cell 1
3. Click **Runtime → Run all**
4. Wait ~5 minutes for everything to load
5. Copy the public URL printed at the end and paste it into Vercel

Your Vercel site will be live for the duration of this Colab session!

In [ ]:
# ============================================================
# CELL 1: Configuration — fill these in!
# ============================================================

# Get your ngrok token from: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = "PASTE_YOUR_NGROK_TOKEN_HERE"

# Get your HF read token from: https://huggingface.co/settings/tokens
HF_TOKEN = "PASTE_YOUR_HF_READ_TOKEN_HERE"

# API key must match your Vercel NEXT_PUBLIC_API_KEY env var
API_KEY = "dev-secret-key"

# Your GGUF model repo on HuggingFace
GGUF_REPO = "Shabuuuuuuuuuuu/GSV-Math-GGUF"

print("✅ Config set. Run the next cells!")

In [ ]:
# ============================================================
# CELL 2: Install llama.cpp (CUDA build for T4 GPU)
# ============================================================
print("Building llama.cpp with CUDA support...")
print("(This takes ~3 minutes the first time)")

import subprocess, os

# Install cmake if needed
subprocess.run(["apt-get", "install", "-y", "-q", "cmake"], check=True)

if not os.path.exists("/content/llama.cpp"):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/ggerganov/llama.cpp",
                    "/content/llama.cpp"], check=True)

os.makedirs("/content/llama.cpp/build", exist_ok=True)
subprocess.run([
    "cmake", "..",
    "-DGGML_CUDA=ON",
    "-DCMAKE_BUILD_TYPE=Release"
], cwd="/content/llama.cpp/build", check=True, capture_output=True)

subprocess.run([
    "cmake", "--build", ".", "--config", "Release",
    "-j", str(os.cpu_count())
], cwd="/content/llama.cpp/build", check=True)

subprocess.run(["cp", "/content/llama.cpp/build/bin/llama-server", "/usr/local/bin/"], check=True)

print("✅ llama.cpp built with CUDA support!")

In [ ]:
# ============================================================
# CELL 3: Download GGUF model files from HuggingFace
# ============================================================
print("Downloading GGUF model files...")
print("(4.4GB base model + 1.3GB vision projector + 155MB LoRA)")

!pip install -q huggingface_hub
from huggingface_hub import hf_hub_download
import os

os.makedirs("/content/models", exist_ok=True)

files = [
    "Qwen2.5-VL-7B-Instruct-Q4_K_M.gguf",
    "mmproj-Qwen2.5-VL-7B-Instruct-f16.gguf",
    "gsv-math-lora.gguf"
]

for f in files:
    dest = f"/content/models/{f}"
    if os.path.exists(dest):
        size = os.path.getsize(dest) / (1024**3)
        print(f"  [CACHED] {f} ({size:.1f}GB)")
    else:
        print(f"  Downloading {f}...")
        hf_hub_download(
            repo_id=GGUF_REPO,
            filename=f,
            local_dir="/content/models",
            token=HF_TOKEN
        )
        size = os.path.getsize(dest) / (1024**3)
        print(f"  Done: {f} ({size:.1f}GB)")

print("\n Model files ready!")
!ls -lh /content/models/

In [ ]:
# ============================================================
# CELL 4: Start llama-server (GPU accelerated)
# ============================================================
import subprocess, time, requests

print("Starting llama-server with GPU acceleration...")

server_proc = subprocess.Popen([
    "llama-server",
    "--model",   "/content/models/Qwen2.5-VL-7B-Instruct-Q4_K_M.gguf",
    "--mmproj",  "/content/models/mmproj-Qwen2.5-VL-7B-Instruct-f16.gguf",
    "--lora",    "/content/models/gsv-math-lora.gguf",
    "--n-gpu-layers", "80",   # Offload all layers to T4 GPU
    "--threads", "2",
    "--ctx-size", "4096",
    "--n-predict", "256",
    "--port", "8081",
    "--host", "127.0.0.1"
], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print("Waiting for model to load (up to 90 seconds)...")
for i in range(90):
    try:
        r = requests.get("http://127.0.0.1:8081/health", timeout=2)
        if r.json().get("status") == "ok":
            print(f"Model loaded in {i+1} seconds!")
            break
    except:
        pass
    if i % 10 == 9:
        print(f"  Still loading... ({i+1}s)")
    time.sleep(1)
else:
    print("ERROR: llama-server did not start in 90s. Check GPU is enabled!")
    server_proc.kill()

In [ ]:
# ============================================================
# CELL 5: Start FastAPI proxy + ngrok tunnel
# ============================================================
!pip install -q fastapi uvicorn pyngrok httpx Pillow

# Write the FastAPI app
app_code = '''
import os, re, base64, io, logging, httpx, threading, uvicorn
from PIL import Image
from fastapi import FastAPI, Request, Depends, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.security.api_key import APIKeyHeader

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)
LLAMA = "http://127.0.0.1:8081"
app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

api_key_header = APIKeyHeader(name="X-API-Key", auto_error=False)
API_KEY = os.environ.get("API_KEY", "dev-secret-key")

async def verify_api_key(api_key: str = Depends(api_key_header)):
    if api_key != API_KEY:
        raise HTTPException(status_code=403, detail="Invalid API Key")

PATTERNS = [
    r"\\\\boxed\\{([^}]*)\\}",
    r"[Ff]inal\\s*[Aa]nswer\\s*[:\\-]?\\s*(.{1,80})",
    r"[Tt]he\\s+answer\\s+is\\s*[:\\-]?\\s*(.{1,80})",
]

def extract_answer(text):
    for p in PATTERNS:
        m = list(re.finditer(p, text, re.IGNORECASE | re.DOTALL))
        if m: return m[-1].group(1).strip()
    idx = text.lower().rfind("answer is")
    if idx != -1: return text[idx+9:].strip().replace(":","").replace(".","").strip()
    words = text.split()
    return words[-1] if words else text

def normalize(ans):
    ans = ans.strip().lower()
    for p in ["x=","y=","z=","a=","b=","c="]:
        if ans.startswith(p): ans = ans[len(p):].strip()
    try:
        f = float(ans)
        return str(int(f)) if f.is_integer() else str(f)
    except: return ans

@app.post("/", dependencies=[Depends(verify_api_key)])
async def solve(request: Request):
    try:
        data = await request.json()
        image_b64 = data.get("image_base64")
        image_url = data.get("image_url")
        question  = data.get("question","")
        if not (image_b64 or image_url) or not question:
            return {"error": "Missing image or question"}
        if image_url and not image_b64:
            async with httpx.AsyncClient(timeout=10) as c:
                r = await c.get(image_url)
                image_b64 = base64.b64encode(r.content).decode()
        # Resize to training resolution
        img = Image.open(io.BytesIO(base64.b64decode(image_b64))).convert("RGB")
        if max(img.size) > 396:
            img.thumbnail((396,396))
            buf = io.BytesIO()
            img.save(buf, format="PNG")
            image_b64 = base64.b64encode(buf.getvalue()).decode()
        msgs = [{"role":"user","content":[
            {"type":"image_url","image_url":{"url":f"data:image/png;base64,{image_b64}"}},
            {"type":"text","text":question}
        ]}]
        async with httpx.AsyncClient(timeout=300) as c:
            r = await c.post(f"{LLAMA}/v1/chat/completions",
                json={"messages":msgs,"max_tokens":256,"temperature":0.7})
            r.raise_for_status()
        raw = r.json()["choices"][0]["message"]["content"]
        ans = normalize(extract_answer(raw))
        logger.info(f"Q: {question[:60]} | A: {ans}")
        return {"answer":ans,"reasoning":raw,"vote_distribution":{ans:1},
                "owl_grounding_score":None,"clip_alignment_score":None,
                "symbolic_check_passed":None,
                "note":"Running on Colab T4 GPU via llama.cpp"}
    except Exception as e:
        logger.error(str(e))
        return {"error": str(e)}

@app.get("/health")
def health(): return {"status":"ok"}

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8080)
'''

with open("/content/backend.py", "w") as f:
    f.write(app_code)

# Set the API key env var
import os
os.environ["API_KEY"] = API_KEY

# Start FastAPI in background thread
import threading, uvicorn
import importlib.util, sys

spec = importlib.util.spec_from_file_location("backend", "/content/backend.py")
mod  = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

def run_api():
    uvicorn.run(mod.app, host="0.0.0.0", port=8080, log_level="error")

t = threading.Thread(target=run_api, daemon=True)
t.start()

import time; time.sleep(2)
import requests
r = requests.get("http://localhost:8080/health")
print(f"FastAPI status: {r.json()}")

# Start ngrok tunnel
from pyngrok import ngrok, conf
conf.get_default().auth_token = NGROK_TOKEN
tunnel = ngrok.connect(8080, "http")
PUBLIC_URL = tunnel.public_url

print(f"""
{'='*60}
YOUR VERCEL BACKEND IS LIVE!
{'='*60}

Public URL: {PUBLIC_URL}

Now go to Vercel and set:
  NEXT_PUBLIC_MODAL_BACKEND_URL = {PUBLIC_URL}

Then redeploy Vercel and your site works!

This session will stay alive until:
  - You close this tab, OR
  - Colab disconnects you (usually 12h)
{'='*60}
""")

In [ ]:
# ============================================================
# CELL 6 (Optional): Keep-alive ping — prevents Colab timeout
# Run this to keep the session alive for up to 12 hours
# ============================================================
import time, requests

print("Keep-alive running. Leave this cell going to prevent Colab timeout.")
print("Stop it (square button) when you're done demoing.")
print()

ping_count = 0
while True:
    try:
        r = requests.get("http://localhost:8080/health", timeout=5)
        ping_count += 1
        if ping_count % 12 == 0:  # Print every minute
            print(f"  Backend alive | {ping_count * 5}s elapsed | URL: {PUBLIC_URL}")
    except Exception as e:
        print(f"WARNING: Backend ping failed: {e}")
    time.sleep(5)